In [2]:
import pandas as pd
from pathlib import Path

# Konfigurasi Visualisasi
pd.set_option('display.max_column',None)

# --- TEKNIK ROBUST: Dynamic Path Resolution ---
# 1. Mengambil lokasi folder saat ini secara dinamis
# Gunakan .cwd() untuk pathlib atau Path(__file__).parent jika di dalam script .py
current_dir = Path.cwd() 

# 2. Navigasi ke Root Project (Olist_Ecommerce_Analytics_Portfolio)
# Berdasarkan struktur folder Anda, dari notebooks/02_logistics_delivery/Research & Dev...
# kita perlu naik 3 tingkat untuk mencapai Root.
project_root = current_dir.parent.parent.parent

# 3. Definisikan path ke file parquet di folder production
data_path = project_root / "data" / "production" / "02_logistics_analytics_data.parquet"

# --- VERIFIKASI & LOADING ---
print(f"🔍 Mencari file di: {data_path}")

if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"✅ Berhasil memuat data!")
    print(f"📊 Shape Data: {df.shape}")
    display(df.head())
else:
    print(f"❌ File TIDAK ditemukan.")
    print(f"TIPS: Pastikan file '02_logistics_analytics_data.parquet' sudah ada di folder production.")

🔍 Mencari file di: c:\Users\etc\OneDrive\Documents\Marketing Data Analyst Portofolio\Olist_Ecommerce_Analytics_Portfolio\data\production\02_logistics_analytics_data.parquet
✅ Berhasil memuat data!
📊 Shape Data: (110182, 24)


,order_id,customer_id,seller_id,product_id,order_status,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,actual_lead_time_days,diff_estimated_vs_actual
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,delivered,8.72,500.0,19.0,8.0,13.0,3149,Sao Paulo,SP,9350.0,maua,SP,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10-06 11:07:15,8.436574,7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,delivered,22.76,400.0,19.0,13.0,19.0,47813,Barreiras,BA,31570.0,belo horizonte,SP,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07-30 03:24:27,13.782037,5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,delivered,19.22,420.0,24.0,19.0,21.0,75265,Vianopolis,GO,14840.0,guariba,SP,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08-13 08:55:23,9.394213,17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,66922902710d126a0e7d26b0e3805106,d0b61bfb1de832b15ba9d266ca96e5b0,delivered,27.20,450.0,30.0,10.0,20.0,59296,Sao Goncalo Do Amarante,RN,31842.0,belo horizonte,MG,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017-11-23 19:45:59,13.208750,12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2c9e548be18521d1c43cde1c582c6de8,65266b2da20d04dbe00c5c2d3bb7859e,delivered,8.72,250.0,51.0,15.0,15.0,9195,Santo Andre,SP,8752.0,mogi das cruzes,SP,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018-02-19 20:31:37,2.873877,9.238171


01. Logistics Descriptive Analysis
Project: 02_logistics_delivery

Author: MDA Specialist

Objective: Melakukan audit kesehatan logistik (Health Check) untuk menetapkan baseline performa pengiriman sebelum memasuki fase diagnostik.

1. Environment Setup & Robust Data Loading
We use pathlib to ensure the script is location-agnostic. Memory optimization is achieved through explicit category casting for low-cardinality strings.

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

def get_project_root() -> Path:
    """Dynamically resolves the project root directory."""
    # Current: notebooks/02_logistics_delivery/Research & Dev/
    return Path.cwd().parent.parent.parent

def load_logistics_data(file_path: Path) -> pd.DataFrame:
    """Loads parquet data with memory optimization."""
    if not file_path.exists():
        raise FileNotFoundError(f"Missing data at {file_path}")
    
    df = pd.read_parquet(file_path)
    
    # Platinum Optimization: Category casting
    cat_cols = ['customer_state', 'seller_state', 'order_status']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

# Execution
ROOT = get_project_root()
DATA_PATH = ROOT / "data" / "production" / "02_logistics_analytics_data.parquet"
df_raw = load_logistics_data(DATA_PATH)

print(f"✅ Data Loaded: {len(df_raw):,} rows")

✅ Data Loaded: 110,182 rows


2. Data Validation & Sanitization (EDA Phase)
We must identify "dirty" logistics data (e.g., items delivered before purchase) and handle outliers dynamically using the 99th percentile to avoid biasing the averages.

In [4]:
def sanitize_logistics(df: pd.DataFrame) -> pd.DataFrame:
    """Performs data cleaning and dynamic outlier removal."""
    # 1. Drop rows with missing delivery dates (cannot calculate lead time)
    df = df.dropna(subset=['order_delivered_customer_date']).copy()
    
    # 2. Logic Check: Delivery cannot happen before purchase
    df = df[df['actual_lead_time_days'] >= 0]
    
    # 3. Dynamic Outlier Removal (99th Percentile)
    # Using 99th percentile is safer than hardcoding '90 days'
    limit = df['actual_lead_time_days'].quantile(0.99)
    df_clean = df[df['actual_lead_time_days'] <= limit].copy()
    
    print(f"🧹 Sanitization Complete: {len(df) - len(df_clean)} outliers removed.")
    return df_clean

df = sanitize_logistics(df_raw)

🧹 Sanitization Complete: 1102 outliers removed.


3. Core Logistics KPIs (Descriptive Phase)
Establishing the baseline for On-Time Delivery (OTD) and Average Lead Time (ALT).

In [ ]:
def calculate_logistics_kpis(df: pd.DataFrame):
    """Calculates operational baseline KPIs."""
    # OTD Rate: Based on diff_estimated_vs_actual from ingestion
    df['is_on_time'] = df['diff_estimated_vs_actual'] >= 0
    otd_rate = df['is_on_time'].mean() * 100
    
    # ALT & Delay Severity
    alt = df['actual_lead_time_days'].mean()
    delay_severity = abs(df[df['is_on_time'] == False]['diff_estimated_vs_actual'].mean())
    
    # Freight Analysis
    avg_freight = df['freight_value'].mean()
    
    print(f"--- LOGISTICS KPI DASHBOARD ---")
    print(f"On-Time Delivery (OTD) Rate : {otd_rate:.2f}%")
    print(f"Average Lead Time (ALT)    : {alt:.2f} Days")
    print(f"Avg Delay Severity         : {delay_severity:.2f} Days")
    print(f"Avg Freight Value          : R$ {avg_freight:.2f}")
    
    return otd_rate, alt

otd_base, alt_base = calculate_logistics_kpis(df)

--- LOGISTICS KPI DASHBOARD ---
On-Time Delivery (OTD) Rate : 92.98%
Average Lead Time (ALT)    : 11.97 Days
Avg Delay Severity         : 6.18 Days
Avg Freight Value          : R$ 19.87


4. Regional Performance Segmentation
Identifying regional bottlenecks is the bridge to Diagnostic Analysis.

In [6]:
# Grouping by state
state_metrics = df.groupby('customer_state').agg({
    'order_id': 'count',
    'actual_lead_time_days': 'mean',
    'is_on_time': 'mean'
}).reset_index()

state_metrics['is_on_time'] *= 100
state_metrics = state_metrics.rename(columns={'order_id': 'order_volume'})

# Identify Top/Bottom 5
top_5_fastest = state_metrics.nsmallest(5, 'actual_lead_time_days')
bottom_5_slowest = state_metrics.nlargest(5, 'actual_lead_time_days')

print("🚀 Most Efficient States (Lead Time):", top_5_fastest['customer_state'].tolist())
print("⚠️ Problematic States (Lead Time):", bottom_5_slowest['customer_state'].tolist())

🚀 Most Efficient States (Lead Time): ['SP', 'MG', 'PR', 'DF', 'RJ']
⚠️ Problematic States (Lead Time): ['AM', 'AP', 'AL', 'PA', 'RR']


C:\Users\etc\AppData\Local\Temp\ipykernel_10900\3321909021.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_metrics = df.groupby('customer_state').agg({


5. High-Impact Visualizations
Interactive charts to communicate health status to stakeholders.

In [7]:
# A. Distribution of Lead Times
fig_hist = px.histogram(df, x="actual_lead_time_days", 
                       title="Logistics Lead Time Distribution",
                       labels={'actual_lead_time_days': 'Days to Deliver'},
                       color_discrete_sequence=['#3366CC'])
fig_hist.add_vline(x=alt_base, line_dash="dash", line_color="red", annotation_text="Average")
fig_hist.show()

# B. Regional Lead Time vs OTD Rate
fig_bar = px.bar(state_metrics.sort_values('actual_lead_time_days'), 
                 x='customer_state', y='actual_lead_time_days',
                 color='is_on_time',
                 title='State Performance: Lead Time vs OTD %',
                 color_continuous_scale='RdYlGn',
                 labels={'actual_lead_time_days': 'Avg Days', 'is_on_time': 'OTD %'})
fig_bar.show()

6. Data Export & Readiness Audit

In [8]:
import datetime

# Professional Comment: Finalizing the descriptive phase by validating data integrity,
# tagging metadata for audit trails, and exporting results for the Diagnostic Phase.

def finalize_descriptive_phase(df_main, df_metrics):
    try:
        # 1. Analytical Integrity Check
        if df_main.empty or df_metrics.empty:
            raise ValueError("DataFrames are empty. Integrity check failed.")
        
        # 2. Metadata Tagging & Versioning
        # Adding audit columns to the metrics for future traceability
        current_time = datetime.datetime.now()
        df_metrics['processed_at'] = current_time
        df_metrics['data_version'] = "v1.0_descriptive"
        
        # 3. Automated Export using Pathlib
        project_root = Path.cwd().parent.parent.parent
        export_path = project_root / "data" / "production" / "03_logistics_descriptive_results.parquet"
        
        # Ensure directory exists
        export_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Exporting with high-performance Snappy compression
        df_metrics.to_parquet(export_path, index=False, compression='snappy')
        
        # 4. Diagnostic Readiness Report
        print("-" * 50)
        print("🏆 DESCRIPTIVE PHASE: 100% COMPLETE")
        print("-" * 50)
        print(f"✅ Export Successful: {export_path.name}")
        print(f"📅 Timestamp      : {current_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"📊 Metrics Scoped : {len(df_metrics)} states analyzed")
        
        # Identifying Top 3 Bottlenecks for the next Diagnostic phase
        bottlenecks = df_metrics.nlargest(3, 'actual_lead_time_days')['customer_state'].tolist()
        print(f"\n🚨 DIAGNOSTIC PRIORITY:")
        print(f"The following states exhibit critical lead time delays: {bottlenecks}")
        print("Action: Initiate '02_logistics_diagnostic_analysis.ipynb' to investigate root causes.")
        print("-" * 50)

    except Exception as e:
        print(f"❌ Critical Error during finalization: {e}")

# Execute Final Process
finalize_descriptive_phase(df, state_metrics)

--------------------------------------------------
🏆 DESCRIPTIVE PHASE: 100% COMPLETE
--------------------------------------------------
✅ Export Successful: 03_logistics_descriptive_results.parquet
📅 Timestamp      : 2026-02-07 23:11:14
📊 Metrics Scoped : 27 states analyzed

🚨 DIAGNOSTIC PRIORITY:
The following states exhibit critical lead time delays: ['AM', 'AP', 'AL']
Action: Initiate '02_logistics_diagnostic_analysis.ipynb' to investigate root causes.
--------------------------------------------------


# 01. Kesimpulan Analisis Deskriptif & Kesiapan Diagnostik

Analisis deskriptif telah berhasil memetakan baseline performa logistik untuk **Project 02: Logistics Delivery**. Berdasarkan 6 tahapan pemrosesan data yang telah dilakukan, berikut adalah ringkasan status operasional saat ini:

### 📈 Ringkasan Performa Logistik
* **Status Proyek**: Tahap Deskriptif 100% Selesai.
* **Cakupan Data**: Analisis mencakup metrik dari **27 negara bagian** (states) di Brasil.
* **Integritas Data**: Semua anomali data telah dibersihkan, dan hasil agregat telah diekspor ke format Parquet (`03_logistics_descriptive_results.parquet`) untuk menjamin konsistensi pada tahap berikutnya.

### 🚨 Prioritas Diagnostik (Bottlenecks)
Berdasarkan metrik rata-rata *Lead Time* tertinggi, tiga wilayah berikut diidentifikasi sebagai hambatan utama yang memerlukan investigasi mendalam:
1. **AM** (Amazonas)
2. **AP** (Amapá)
3. **AL** (Alagoas)

---

# 🚀 Langkah Selanjutnya (Next Step)

Tahap deskriptif telah menjawab **"APA"** yang terjadi dalam performa logistik kita. Selanjutnya, kita akan beranjak ke notebook **`02_logistics_diagnostic_analysis.ipynb`** untuk menjawab **"MENGAPA"** keterlambatan tersebut terjadi secara sistematis.

**Fokus Investigasi Diagnostik:**
* **Korelasi Jarak & Rute**: Menganalisis apakah keterlambatan di wilayah prioritas (AM, AP, AL) murni disebabkan oleh jarak geografis atau inefisiensi kurir tertentu.
* **Analisis Berat & Volume**: Menguji apakah dimensi produk memengaruhi kecepatan pengiriman di wilayah bottleneck tersebut.
* **Perbandingan Estimasi vs Aktual**: Mengidentifikasi apakah masalah utama terletak pada operasional logistik atau pada akurasi algoritma estimasi pengiriman.

---
**Timestamp Laporan:** 2026-02-07 23:11:14 | **Versi Data:** v1.0_descriptive